In [ ]:
import pandas as pd
import numpy as np
import os 

In [ ]:
os.chdir("/Users/kanhapandey/Projects/SIH_PAIMANA")

## 1. Load all four files

In [ ]:
files = {
    "monthly_ongoing":   "data/cleaned/monthly_ongoing_projects_clean.csv",
    "monthly_completed": "data/cleaned/monthly_completed_projects_clean.csv",
    "quarterly_completed": "data/cleaned/quaterly_completed_projects_clean.csv",
    "quarterly_frozen":  "data/cleaned/quaterly_frozen_or_deleted_projects_clean.csv",
}
 
dfs = {name: pd.read_csv(path) for name, path in files.items()}

## 2. Define target dtypes for every column

In [ ]:
date_cols = ["date_of_approval", "start_date", "actual_doc", "target_doc", "revised_doc"]
float_cols = ["original_cost_cr", "revised_cost_cr",
              "cumulative expenditure in rs. crore", "physical progress (in percentage)"]
str_cols = ["project_code", "project_name", "agency", "state",
            "ministry", "sector", "status"]
 
for name, df in dfs.items():
    # dates -> consistent datetime64[ns]
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    # numerics -> float64
    for c in float_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    # text -> plain object/str (drops pandas 'string' dtype inconsistency)
    for c in str_cols:
        df[c] = df[c].astype("object").where(df[c].notna(), pd.NA)

## 3. Sanity check: confirm columns + order match before concat

In [ ]:
ref_cols = list(dfs["monthly_ongoing"].columns)
for name, df in dfs.items():
    assert list(df.columns) == ref_cols, f"Column mismatch in {name}: {list(df.columns)}"

## 4. Concat

In [ ]:
merged = pd.concat(dfs.values(), ignore_index=True)

## 5. Post-merge checks

In [ ]:
print("Final shape:", merged.shape)
print("\nDtypes:\n", merged.dtypes)
print("\nStatus counts:\n", merged["status"].value_counts())
print("\nDuplicate project_code + status pairs:",
      merged.duplicated(subset=["project_code", "status"]).sum())
print("\nNull counts:\n", merged.isna().sum())

## 6. Preparing the merged dataframe for Machine Learning

In [ ]:
merged

In [ ]:
merged.isna().sum()

#### Missingness flag columns

In [ ]:
columns = [
    "date_of_approval",
    "start_date",
    "actual_doc",
    "target_doc",
    "revised_doc",
    "revised_cost_cr",
    "ministry"
]

for col in columns:
    merged[f"{col}_is_missing"] = merged[col].isna().astype(int)

#### Derived TARGET columns

In [ ]:
merged["cost_overrun_abs_cr"] = merged["revised_cost_cr"] - merged["original_cost_cr"]
merged["cost_overrun_pct"] = merged["cost_overrun_abs_cr"] / merged["original_cost_cr"]

# True delay: only defined where actual completion date exists
merged["delay_actual_days"] = merged["actual_doc"] - merged["target_doc"]

# Proxy delay: schedule-revision-based, defined wherever revised_doc exists
merged["delay_proxy_days"] = merged["revised_doc"] - merged["target_doc"]
merged

#### Expenditure vs. cost / progress signal columns

In [ ]:
exp_col = "cumulative expenditure in rs. crore"
progress_col = "physical progress (in percentage)"
 
merged["expenditure_to_original_cost_ratio"] = merged[exp_col] / merged["original_cost_cr"]
merged["expenditure_to_revised_cost_ratio"] = merged[exp_col] / merged["revised_cost_cr"]
merged["expenditure_exceeds_revised_cost"] = merged[exp_col] > merged["revised_cost_cr"]
 
# Mismatch score: how far expenditure ratio is ahead of / behind delivered progress
# (progress is 0-100, so we normalize to 0-1 for comparability with expenditure ratio)
merged["expenditure_progress_mismatch"] = (
    merged["expenditure_to_original_cost_ratio"] - (merged[progress_col] / 100)
)
 
# Convenience flag using the threshold discussed (progress <=20% and expenditure >=80% of
# original cost). Threshold is a default suggestion, not a fixed rule - documented in the
# data dictionary so the ML team can adjust it.
merged["high_spend_low_progress_flag"] = (
    (merged[progress_col] <= 20) & (merged["expenditure_to_original_cost_ratio"] >= 0.8)
)

#### Data-quality flags (surfaced, not auto-corrected)

In [ ]:
merged["negative_expenditure_flag"] = merged[exp_col] < 0
merged["negative_cost_overrun_flag"] = merged["cost_overrun_abs_cr"] < 0  # revised < original (cost reduced)

## 7. Save

In [ ]:
merged.to_csv("data/cleaned/merged_projects.csv", index=False)

## 8. Some Information

In [ ]:
print("Final shape:", merged.shape)
print("\nQuick sanity counts:")
print("  rows with cost_overrun_pct computable:", merged["cost_overrun_pct"].notna().sum())
print("  rows with delay_actual_days computable:", merged["delay_actual_days"].notna().sum())
print("  rows with delay_proxy_days computable:", merged["delay_proxy_days"].notna().sum())
print("  high_spend_low_progress_flag True:", merged["high_spend_low_progress_flag"].sum())
print("  negative_expenditure_flag True:", merged["negative_expenditure_flag"].sum())
print("  negative_cost_overrun_flag True:", merged["negative_cost_overrun_flag"].sum())